In [32]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
import sys

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

cat_features = ['city', 'district_type', 'fitting_rooms_count', 'entrance_type', 'floor', 'parking_nearby', 'competitor_pvz_count_500m']
features = train.columns.drop(['id', 'daily_orders'])

X = train[features]
y = train['daily_orders']

In [33]:
import json
import sys
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

# 1. Загрузка
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

cat_features = [
    'city', 'district_type', 'fitting_rooms_count', 'entrance_type',
    'floor', 'parking_nearby', 'competitor_pvz_count_500m'
]
features = train.columns.drop(['id', 'daily_orders'])

X = train[features]
y = train['daily_orders'].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)

# 2. Кросс-валидация (KFold как в стартовом скрипте)
splits = list(KFold(4, shuffle=True, random_state=0).split(train))
oof = np.zeros((len(train), 3))

for a, b in splits:
    X_tr, y_tr = X.iloc[a], y[a]
    X_val = X.iloc[b]
    for i, q in enumerate(QUANTILES):
        model = CatBoostRegressor(
            loss_function=f'Quantile:alpha={q}',
            iterations=100,
            learning_rate=0.05,
            depth=6,
            random_seed=42,
            verbose=0
        )
        model.fit(X_tr, y_tr, cat_features=cat_features)
        oof[b, i] = model.predict(X_val)

# сортируем квантили, чтобы p10 <= p50 <= p90
oof = np.sort(oof, axis=1)

# считаем метрику по фолдам
folds = [
    100 * quantile_quality(
        y[b], oof[b, 0], oof[b, 1], oof[b, 2], train["city"].iloc[b]
    )
    for _, b in splits
]

# 3. Обучение на полном трейне для теста
test_preds = np.zeros((len(test), 3))
for i, q in enumerate(QUANTILES):
    model = CatBoostRegressor(
        loss_function=f'Quantile:alpha={q}',
        iterations=100,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=0
    )
    model.fit(X, y, cat_features=cat_features)
    test_preds[:, i] = model.predict(test[features])

test_preds = np.sort(test_preds, axis=1)
test_preds = np.clip(test_preds, 0, 5000)

# 4. submission.csv
submission = pd.DataFrame({
    "id": test["id"],
    "predicted_daily_orders_p10": test_preds[:, 0],
    "predicted_daily_orders_p50": test_preds[:, 1],
    "predicted_daily_orders_p90": test_preds[:, 2]
})
submission.to_csv("submission.csv", index=False)

# 5. cv.json
cv = cv_json("catboost_baseline", [f / 100 for f in folds])
print(json.dumps(cv))

{"name": "catboost_baseline", "metric": "quantile_quality_x100", "folds": [70.19, 72.16, 70.93, 70.93], "cv": 71.05, "std": 0.71}


In [ ]:
folds